<a href="https://colab.research.google.com/github/kyuho11488/kyuho11488/blob/main/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D%EC%8B%A4%EC%8A%B5_11%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C_2021148025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 문제1 : 기본 CNN 모델 구현 및 평가하기

# 라이브러리 불러오기
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

#  MNIST 데이터 불러오기 및 전처리 (0~255 -> 0~1)
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train[..., None] / 255.0 # (28,28) -> (28,28,1)
x_test = x_test[..., None] / 255.0
y_train = to_categorical(y_train) # One-hot 인코딩
y_test = to_categorical(y_test)

# 기본 CNN 모델 정의하기(Conv -> MaxPool -> Conv -> MaxPool -> FC -> 출력하기)
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation = 'relu', input_shape = (28, 28, 1)), # 첫 번째 합성곱
    layers.MaxPooling2D((2,2)), # 풀링

    layers.Conv2D(64, (3,3), activation = 'relu'), # 두 번째 합성곱
    layers.MaxPooling2D((2,2)),

    layers.Flatten(), # 평탄화하기
    layers.Dense(128, activation = 'relu'), # FC1
    layers.Dense(10, activation = 'softmax') # 출력층 (10개 클래스)
])

# 컴파일 및 학습하기
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(x_train, y_train, epochs = 3, batch_size = 128, validation_data = (x_test,y_test), verbose = 2)
# 출력이 너무 오래걸려서 출력 속도 향상을 위해 다음과 같은 부분을 수정했습니다.
# 1. epochs를 5에서 3으로 줄였습니다. (결과 확인에 충분합니다.)
# 배치 크기(batch_size)를 64에서 128로 늘렸습니다. (학습 속도 향상)

# 평가하기
test_loss, test_acc = model.evaluate(x_test, y_test, verbose = 0)
print(f"기본 CNN 모델 정확도 : {test_acc : .4f}")

Epoch 1/3
469/469 - 48s - 102ms/step - accuracy: 0.9406 - loss: 0.2068 - val_accuracy: 0.9838 - val_loss: 0.0529
Epoch 2/3
469/469 - 43s - 91ms/step - accuracy: 0.9827 - loss: 0.0560 - val_accuracy: 0.9860 - val_loss: 0.0426
Epoch 3/3
469/469 - 82s - 175ms/step - accuracy: 0.9888 - loss: 0.0377 - val_accuracy: 0.9884 - val_loss: 0.0347
기본 CNN 모델 정확도 :  0.9884


In [5]:
# 문제2 : 개선 CNN 모델(Conv3 + FC2 추가) 및 정확도 비교하기

# 개선 CNN 모델 구성하기(Conv2D 층과 FC 층을 각각 1개씩 추가하기)
model2 = models.Sequential([
    layers.Conv2D(32, (3,3), activation = 'relu', input_shape = (28, 28, 1)), # conv1
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation = 'relu'), # conv2
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation = 'relu'), # conv3 추가하기

    layers.Flatten(),
    layers.Dense(128, activation = 'relu'), # FC1
    layers.Dense(64, activation = 'relu'), # FC2 추가하기
    layers.Dense(10, activation = 'softmax') # 출력층
])

# 컴파일 및 학습하기
model2.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
model2.fit(x_train, y_train, epochs = 3, batch_size = 128, validation_data = (x_test, y_test), verbose = 2)
# 이 문제 역시 출력 속도가 매우 느려서 위 문제와 같이 똑같은 부분을 수정했습니다.
# 1. epochs를 5에서 3으로 줄였습니다. (결과 확인에 충분합니다.)
# 2. 배치 크기(batch_size)를 64에서 128로 늘렸습니다. (학습 속도 향상)

# 평가하기
test_loss2, test_acc2 = model2.evaluate(x_test, y_test, verbose = 0)
print(f"개선 CNN 모델 정확도 : {test_acc2 : .4f}")

Epoch 1/3
469/469 - 52s - 111ms/step - accuracy: 0.9317 - loss: 0.2236 - val_accuracy: 0.9844 - val_loss: 0.0505
Epoch 2/3
469/469 - 81s - 172ms/step - accuracy: 0.9841 - loss: 0.0537 - val_accuracy: 0.9891 - val_loss: 0.0352
Epoch 3/3
469/469 - 83s - 176ms/step - accuracy: 0.9874 - loss: 0.0397 - val_accuracy: 0.9886 - val_loss: 0.0337
개선 CNN 모델 정확도 :  0.9886


In [6]:
# 결과 비교 출력하기(두 모델의 성능 비교 출력하기)
print("\n 정확도 비교")
print(f"기본 모델 : {test_acc : .4f}")
print(f"개선 모델 : {test_acc2 : .4f}")


 정확도 비교
기본 모델 :  0.9884
개선 모델 :  0.9886


개선 모델이 기존 모델보다 항상 정확도가 높게 나오지 않는 이유

모델 복잡도가 높아졌지만 데이터가 충분하지 않으면 오히려 불리하다.
- Conv3, FC2 레이어를 추가하면 모델의 파리미터 수가 증가하고 학습이 더 어려워질 수도 있습니다.
- 이런 경우, 오히려 과적합(overfitting) 위험이 생기며, 테스트 정확도가 기존 모델보다 낮아질 수 있습니다.
- 특히, epochs 수가 작으면, 복잡도 증가가 항상 이득으로 작용하지 않습니다.(만약 epochs 수를 10 이상으로 늘리면, 출력 시간이 더 오래 걸리면서 개선 모델이 기존 모델보다 항상 정확도가 높게 나오는 것도 아닙니다.)